In [1]:
import pandas as pd
import json

In [2]:
file_path = '../data/miband/20260406_8278074507_MiFitness_ams1_data_copy/20260406_8278074507_MiFitness_hlth_center_aggregated_fitness_data.csv'
df_raw = pd.read_csv(file_path)

In [3]:
df_sleep = df_raw[(df_raw['Key'] == 'sleep') & (df_raw['Tag'] == 'daily_report') ].copy()

# Parse JSON values and normalize into separate columns
sleep_data_list = [json.loads(val) for val in df_sleep['Value']]
df_sleep_normalized = pd.json_normalize(sleep_data_list)

# Flatten nested stress_scale columns
sleep_scale_cols = [col for col in df_sleep_normalized.columns if col.startswith('segment_detail.')]
for col in sleep_scale_cols:
    new_col_name = col.replace('stress_scale.', '')
    df_sleep_normalized[new_col_name] = df_sleep_normalized[col]
    df_sleep_normalized = df_sleep_normalized.drop(col, axis=1)

# Combine with original metadata (Time, Uid, etc.)
df_sleep_result = df_sleep[['Uid', 'Sid', 'Time', 'UpdateTime']].reset_index(drop=True).join(df_sleep_normalized)

df_sleep_result

,Uid,Sid,Time,UpdateTime,total_duration,sleep_rem_duration,sleep_nap_duration,total_turn_over,avg_hr,avg_spo2,...,total_body_move,awake_count,max_hr,total_snore_disturb,segment_details,sleep_deep_duration,min_hr,min_spo2,max_spo2,sleep_algorithm_version
0,8278074507,default,1736899200,1736933977,428,125,0,0,56,0,...,0,2,80,0,"[{'awake_count': 2, 'wake_up_time': 1736930100...",136,48,NaN,NaN,NaN
1,8278074507,default,1736985600,1737032483,515,119,65,0,54,0,...,0,1,79,0,"[{'awake_count': 1, 'sleep_light_duration': 22...",105,43,NaN,NaN,NaN
2,8278074507,default,1737072000,1737115598,496,121,0,0,57,0,...,0,2,99,0,"[{'sleep_rem_duration': 121, 'wake_up_time': 1...",128,47,NaN,NaN,NaN
3,8278074507,default,1737158400,1737279494,523,121,0,0,50,0,...,0,0,79,0,"[{'duration': 523, 'sleep_light_duration': 246...",156,43,NaN,NaN,NaN
4,8278074507,default,1737244800,1737279494,432,108,0,0,50,0,...,0,1,61,0,"[{'timezone': 4, 'sleep_awake_duration': 4, 'a...",152,40,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
221,8278074507,default,1775088000,1775210502,470,51,0,0,52,97,...,0,1,76,0,"[{'bedtime': 1775074980, 'wake_up_time': 17751...",162,46,95.0,99.0,0.0
222,8278074507,default,1775174400,1775220666,824,158,78,0,56,98,...,0,1,84,0,"[{'sleep_deep_duration': 234, 'sleep_light_dur...",234,42,95.0,99.0,0.0
223,8278074507,default,1775260800,1775300269,779,180,0,0,52,98,...,0,1,81,0,"[{'avg_hr': 52, 'wake_up_time': 1775294700, 't...",289,41,95.0,99.0,0.0
224,8278074507,default,1775347200,1775382390,699,159,0,0,50,98,...,0,1,77,0,"[{'wake_up_time': 1775376000, 'max_spo2': 99, ...",231,41,94.0,99.0,0.0


In [4]:
# Convert Time and UpdateTime columns from epoch to datetime
df_sleep_result['Time'] = pd.to_datetime(df_sleep_result['Time'], unit='s')
df_sleep_result['UpdateTime'] = pd.to_datetime(df_sleep_result['UpdateTime'], unit='s')

df_sleep_result

,Uid,Sid,Time,UpdateTime,total_duration,sleep_rem_duration,sleep_nap_duration,total_turn_over,avg_hr,avg_spo2,...,total_body_move,awake_count,max_hr,total_snore_disturb,segment_details,sleep_deep_duration,min_hr,min_spo2,max_spo2,sleep_algorithm_version
0,8278074507,default,2025-01-15,2025-01-15 09:39:37,428,125,0,0,56,0,...,0,2,80,0,"[{'awake_count': 2, 'wake_up_time': 1736930100...",136,48,NaN,NaN,NaN
1,8278074507,default,2025-01-16,2025-01-16 13:01:23,515,119,65,0,54,0,...,0,1,79,0,"[{'awake_count': 1, 'sleep_light_duration': 22...",105,43,NaN,NaN,NaN
2,8278074507,default,2025-01-17,2025-01-17 12:06:38,496,121,0,0,57,0,...,0,2,99,0,"[{'sleep_rem_duration': 121, 'wake_up_time': 1...",128,47,NaN,NaN,NaN
3,8278074507,default,2025-01-18,2025-01-19 09:38:14,523,121,0,0,50,0,...,0,0,79,0,"[{'duration': 523, 'sleep_light_duration': 246...",156,43,NaN,NaN,NaN
4,8278074507,default,2025-01-19,2025-01-19 09:38:14,432,108,0,0,50,0,...,0,1,61,0,"[{'timezone': 4, 'sleep_awake_duration': 4, 'a...",152,40,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
221,8278074507,default,2026-04-02,2026-04-03 10:01:42,470,51,0,0,52,97,...,0,1,76,0,"[{'bedtime': 1775074980, 'wake_up_time': 17751...",162,46,95.0,99.0,0.0
222,8278074507,default,2026-04-03,2026-04-03 12:51:06,824,158,78,0,56,98,...,0,1,84,0,"[{'sleep_deep_duration': 234, 'sleep_light_dur...",234,42,95.0,99.0,0.0
223,8278074507,default,2026-04-04,2026-04-04 10:57:49,779,180,0,0,52,98,...,0,1,81,0,"[{'avg_hr': 52, 'wake_up_time': 1775294700, 't...",289,41,95.0,99.0,0.0
224,8278074507,default,2026-04-05,2026-04-05 09:46:30,699,159,0,0,50,98,...,0,1,77,0,"[{'wake_up_time': 1775376000, 'max_spo2': 99, ...",231,41,94.0,99.0,0.0
